# CV Project 1 — Energy Price and Load Forecasting

This notebook is the **deterministic / intraday forecasting** project behind the CV statement:

> Developed short-term electricity price and load forecasting models using LASSO, LSTM and quantile regression, improving MAE versus a naïve benchmark across short intraday horizons and using the forecasts for trading-signal / imbalance-risk interpretation.

In the current repository, the clean mapping is:

| CV wording | Repository implementation |
|---|---|
| LASSO | `point-ARX`: a linear ARX model with L1 regularization |
| LSTM | optional sequence model, registered as `model_class="SEQ"` and implemented in `tools/models/SeqModel.py` |
| Quantile regression | `QR-DNN`: quantile-regression DNN; the median `q_0.5` is scored as a point forecast |
| Naïve benchmark | previous-day same-hour price |
| Intraday horizons | first four forecast hours of each 24h forecast block: h+1, h+2, h+3, h+4 |

The notebook uses the existing recalibration engine wherever possible. It creates CV-specific experiment configs locally if they are missing, without overwriting your existing experiments unless you set `OVERWRITE_CONFIGS=True`.


## 1. Setup

In [ ]:
from pathlib import Path
import os
import sys
import json
import pickle
from typing import Dict, List

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Run this notebook from either the repo root or Electricity_Price_Load_Forecasting_3.
PROJECT_DIR = Path.cwd()
if PROJECT_DIR.name != "Electricity_Price_Load_Forecasting_3":
    candidate = PROJECT_DIR / "Electricity_Price_Load_Forecasting_3"
    if candidate.exists():
        PROJECT_DIR = candidate

os.chdir(PROJECT_DIR)
sys.path.insert(0, str(PROJECT_DIR))

from tools.PrTSF_Recalib_tools import PrTsfRecalibEngine, load_data_model_configs

PROJECT_DIR

## 2. Controls

`RUN_RECALIBRATION=False` means: use saved results if they exist.  
If a result file is missing, set `RUN_RECALIBRATION=True` for that experiment.

For interview preparation, you usually only need to run the saved results and inspect the tables.

In [ ]:
TASK_NAME = "EM_price"

RUN_RECALIBRATION = False
OVERWRITE_CONFIGS = False

# Use a short CV-specific run id so we do not touch the original experiment runs.
CV_RUN_ID = "cv_recalib_may_2017"

# Keep the same out-of-sample month as the probabilistic project.
OOS_START = {"y": 2017, "m": 5, "d": 1}
OOS_END = {"y": 2017, "m": 5, "d": 31}

OUTPUT_DIR = PROJECT_DIR / "experiments" / "tasks" / TASK_NAME / "cv_project_1_intraday_summary"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_DIR

## 3. Create CV experiment configs if missing

This cell does not retrain anything. It only ensures that the config files needed by the existing engine are present.

The sequence model is labelled `SEQ` in the code to avoid changing the original model naming convention, but internally it uses a recurrent Keras sequence layer.

In [ ]:
def write_json_if_missing(path: Path, content: Dict, overwrite: bool = False):
    path.parent.mkdir(parents=True, exist_ok=True)
    if path.exists() and not overwrite:
        print(f"exists: {path}")
        return
    with open(path, "w") as f:
        json.dump(content, f, indent=4)
    print(f"written: {path}")


def point_config(model_class: str, optuna_m: str, target_alpha=None):
    return {
        "data_config": {
            "dataset_name": "EM_market__2015-01-03__2017-12-31.csv",
            "idx_start_train": {"y": 2015, "m": 1, "d": 3},
            "idx_start_oos_preds": OOS_START,
            "idx_end_oos_preds": OOS_END,
            "keep_past_train_samples": False,
            "steps_lag_win": 7,
            "pred_horiz": 24,
            "preprocess": "StandardScaler",
            "shuffle_mode": "none",
            "num_vali_samples": 100,
        },
        "model_config": {
            "PF_method": "point",
            "model_class": model_class,
            "optuna_m": optuna_m,
            "target_alpha": target_alpha or [],
            "max_epochs": 800,
            "batch_size": 64,
            "patience": 20,
            "num_ense": 1,
        },
    }


# LASSO-style ARX config.
write_json_if_missing(
    PROJECT_DIR / "experiments" / "tasks" / TASK_NAME / "point-ARX" / CV_RUN_ID / "exper_configs.json",
    point_config(model_class="ARX", optuna_m="grid_search"),
    overwrite=OVERWRITE_CONFIGS,
)
write_json_if_missing(
    PROJECT_DIR / "experiments" / "tasks" / TASK_NAME / "point-ARX" / CV_RUN_ID / "tuned_hyperp-grid_search.json",
    {"l1": 1e-7, "lr": 1e-3},
    overwrite=OVERWRITE_CONFIGS,
)

# Optional recurrent sequence model config. This requires tools/models/SeqModel.py and model_class="SEQ".
write_json_if_missing(
    PROJECT_DIR / "experiments" / "tasks" / TASK_NAME / "point-SEQ" / CV_RUN_ID / "exper_configs.json",
    point_config(model_class="SEQ", optuna_m="random"),
    overwrite=OVERWRITE_CONFIGS,
)
write_json_if_missing(
    PROJECT_DIR / "experiments" / "tasks" / TASK_NAME / "point-SEQ" / CV_RUN_ID / "tuned_hyperp-random.json",
    {
        "hidden_size": 64,
        "n_hidden_layers": 1,
        "lr": 1e-3,
        "activation": "tanh",
        "dense_activation": "softplus",
    },
    overwrite=OVERWRITE_CONFIGS,
)


## 4. Helper functions

In [ ]:
def result_pickle_path(task_name: str, exper_setup: str, run_id: str, optuna_m: str) -> Path:
    return (
        PROJECT_DIR
        / "experiments"
        / "tasks"
        / task_name
        / exper_setup
        / run_id
        / "results"
        / f"recalib_test_results-tuned-{optuna_m}.p"
    )


def load_or_run_experiment(label: str, exper_setup: str, run_id: str, hyper_mode: str = "load_tuned"):
    configs = load_data_model_configs(task_name=TASK_NAME, exper_setup=exper_setup, run_id=run_id)
    optuna_m = configs["model_config"]["optuna_m"]
    result_path = result_pickle_path(TASK_NAME, exper_setup, run_id, optuna_m)

    if result_path.exists() and not RUN_RECALIBRATION:
        print(f"[{label}] loading saved results: {result_path}")
        with open(result_path, "rb") as f:
            return pickle.load(f), configs

    if not RUN_RECALIBRATION:
        print(f"[{label}] missing results: {result_path}")
        print("Set RUN_RECALIBRATION=True and rerun this cell to train/recalibrate.")
        return None, configs

    print(f"[{label}] running recalibration. This may take time.")
    engine = PrTsfRecalibEngine(
        data_configs=configs["data_config"],
        model_configs=configs["model_config"],
    )
    model_hyperparams = engine.get_model_hyperparams(
        method=hyper_mode,
        optuna_m=configs["model_config"]["optuna_m"],
    )
    preds = engine.run_recalibration(
        model_hyperparams=model_hyperparams,
        plot_history=False,
        plot_weights=False,
    )
    return preds, configs


def build_target_quantiles(target_alpha: List[float]) -> List[float]:
    qs = [0.5]
    for alpha in target_alpha:
        qs.append(alpha / 2)
        qs.append(1 - alpha / 2)
    return sorted(qs)


def get_target_col(df: pd.DataFrame) -> str:
    if TASK_NAME in df.columns:
        return TASK_NAME
    if f"TARG__{TASK_NAME}" in df.columns:
        return f"TARG__{TASK_NAME}"
    raise KeyError(f"Cannot find target column in {list(df.columns)}")


def get_median_col(df: pd.DataFrame):
    if 0.5 in df.columns:
        return 0.5
    if "0.5" in df.columns:
        return "0.5"
    raise KeyError("Cannot find median/point forecast column 0.5")


def add_naive_previous_day_same_hour(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    target_col = get_target_col(out)
    out["naive_prev_day_same_hour"] = out[target_col].shift(24)
    return out


def score_horizons_1_to_4(df: pd.DataFrame, model_label: str) -> pd.DataFrame:
    target_col = get_target_col(df)
    pred_col = get_median_col(df)
    work = add_naive_previous_day_same_hour(df)
    work["horizon_h"] = (np.arange(len(work)) % 24) + 1
    work = work.dropna(subset=["naive_prev_day_same_hour"])
    work = work[work["horizon_h"].between(1, 4)]

    rows = []
    for h, g in work.groupby("horizon_h"):
        mae_model = np.mean(np.abs(g[target_col] - g[pred_col]))
        mae_naive = np.mean(np.abs(g[target_col] - g["naive_prev_day_same_hour"]))
        improvement = 100 * (mae_naive - mae_model) / mae_naive if mae_naive != 0 else np.nan
        rows.append({
            "model": model_label,
            "horizon_h": int(h),
            "mae_model": mae_model,
            "mae_naive": mae_naive,
            "improvement_vs_naive_pct": improvement,
        })
    return pd.DataFrame(rows)


def trading_signal_summary(df: pd.DataFrame, model_label: str, threshold: float = 5.0) -> pd.DataFrame:
    target_col = get_target_col(df)
    pred_col = get_median_col(df)
    work = add_naive_previous_day_same_hour(df).dropna(subset=["naive_prev_day_same_hour"]).copy()
    work["forecast_edge"] = work[pred_col] - work["naive_prev_day_same_hour"]
    work["realized_edge"] = work[target_col] - work["naive_prev_day_same_hour"]
    work["signal"] = np.where(work["forecast_edge"] > threshold, 1,
                       np.where(work["forecast_edge"] < -threshold, -1, 0))
    traded = work[work["signal"] != 0].copy()
    if traded.empty:
        return pd.DataFrame([{
            "model": model_label,
            "threshold": threshold,
            "n_signals": 0,
            "hit_rate": np.nan,
            "avg_realized_edge_on_signal": np.nan,
        }])
    traded["hit"] = np.sign(traded["realized_edge"]) == traded["signal"]
    return pd.DataFrame([{
        "model": model_label,
        "threshold": threshold,
        "n_signals": len(traded),
        "hit_rate": traded["hit"].mean(),
        "avg_realized_edge_on_signal": traded["realized_edge"].mean(),
    }])


## 5. Load or run deterministic and quantile-regression models

This notebook scores:

1. **LASSO-ARX**: point forecast from L1-regularized ARX.
2. **Recurrent sequence model**: optional sequence / LSTM-style model, if available.
3. **QR-DNN median**: the quantile-regression model from the probabilistic project, scored through its median forecast.

The point of this first CV project is MAE improvement versus naïve, not Pinball/Winkler.


In [ ]:
experiments = [
    {"label": "LASSO-ARX", "setup": "point-ARX", "run_id": CV_RUN_ID, "hyper_mode": "load_tuned"},
    {"label": "LSTM/sequence", "setup": "point-SEQ", "run_id": CV_RUN_ID, "hyper_mode": "load_tuned"},
    {"label": "Quantile regression median (QR-DNN)", "setup": "QR-DNN", "run_id": "recalib_opt_random_1_2", "hyper_mode": "load_tuned"},
]

predictions = {}
configs = {}
for exp in experiments:
    try:
        preds, conf = load_or_run_experiment(
            label=exp["label"],
            exper_setup=exp["setup"],
            run_id=exp["run_id"],
            hyper_mode=exp["hyper_mode"],
        )
        if preds is not None:
            predictions[exp["label"]] = preds
        configs[exp["label"]] = conf
    except Exception as e:
        print(f"[{exp['label']}] skipped because of error: {e}")

list(predictions.keys())

## 6. MAE versus naïve benchmark across h+1 to h+4

In [ ]:
mae_tables = []
for label, preds in predictions.items():
    mae_tables.append(score_horizons_1_to_4(preds, label))

mae_summary = pd.concat(mae_tables, ignore_index=True).sort_values(["horizon_h", "mae_model"])
mae_summary.to_csv(OUTPUT_DIR / "mae_vs_naive_horizons_1_to_4.csv", index=False)
mae_summary

## 7. Trading-signal / imbalance-risk interpretation

In [ ]:
signal_tables = []
for label, preds in predictions.items():
    signal_tables.append(trading_signal_summary(preds, label, threshold=5.0))

signal_summary = pd.concat(signal_tables, ignore_index=True)
signal_summary.to_csv(OUTPUT_DIR / "trading_signal_summary.csv", index=False)
signal_summary

## 8. Visual check

In [ ]:
best_model = mae_summary.groupby("model")["mae_model"].mean().sort_values().index[0]
plot_df = add_naive_previous_day_same_hour(predictions[best_model]).dropna().iloc[:24*5]
target_col = get_target_col(plot_df)
pred_col = get_median_col(plot_df)

ax = plot_df[[target_col, pred_col, "naive_prev_day_same_hour"]].plot(figsize=(14, 5))
ax.set_title(f"{best_model}: realized price vs forecast vs naïve")
ax.set_ylabel("Electricity price")
ax.grid(True)
plt.show()

## 9. Interview explanation

Use this version:

> This was the deterministic short-horizon forecasting project. I built a pipeline to forecast short-term electricity prices using historical prices, future load/renewable forecasts and calendar features. I compared a LASSO-style ARX baseline, a recurrent sequence model and a quantile-regression DNN. For the deterministic comparison I evaluated the median/point forecast using MAE against a naïve previous-day same-hour benchmark across the first four forecast horizons. I then translated forecast edges versus the naïve benchmark into simple trading signals to understand directional accuracy and imbalance-risk exposure.
